# Five-DOF tip control with an external magnet

Exact final notebook selected by the project author. Tip task DOFs and magnet action dimensions are distinct.

Selected source excerpts from the original research notebook. Outputs, execution counts, widget state and duplicate plotting trials are removed. Definitions are kept as code; configuration, training launches and hardware calls are displayed as reference text so Run All cannot start training or connect to a robot. This is a review extract, not a complete reproducibility package. See README.md for omitted dependencies.


In [ ]:
import os
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
device = torch.device("cpu")


## Source cell 3 (original zero-based index 2)


Reference only; not executed.

```python
import os
import torch
import random
import numpy as np
from tqdm import tqdm
import torch.nn as nn
from utils import MatlabEnv_dataset
import torch.optim as optim
from scipy.io import loadmat
from collections import deque

from utils import euler_zyx_to_quat

# Configuration
current_dir = os.getcwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Path configurations
EXP_NAME = '2000 eps-min v w-Rand-cube-13'
PATHS = {
    # Weights
    'best_weights': os.path.join(current_dir, 'Weights', EXP_NAME, 'Best_weights'),
    'last_weights': os.path.join(current_dir, 'Weights', EXP_NAME, 'Last_weights'),
    'latest_models': os.path.join(current_dir, 'Weights', EXP_NAME, 'Latest_weights'),
    # Variables
    'variables': os.path.join(current_dir, 'Variables', EXP_NAME),

    # Renders
    'renders_train': os.path.join(current_dir, 'Renders', EXP_NAME, 'Train_DRL'),
    'render_tests': os.path.join(current_dir, 'Renders', EXP_NAME, 'Test_DRL'),

    # Plots
    'plots_learning': os.path.join(current_dir, 'Plots', EXP_NAME, 'Learning_Curves'),
    'plots_test': os.path.join(current_dir, 'Plots', EXP_NAME, 'Tests'),
    'plots_generated_points': os.path.join(current_dir, 'Plots', EXP_NAME, 'Generated_Points'),
    'plots_train': os.path.join(current_dir, 'Plots', EXP_NAME, 'Trains'),

    # MATLAB workspaces
    'matlab_workspaces': os.path.abspath(os.path.join(current_dir, '..', '..', '2) Matlab_Codes', '4) Workspaces', 'Cartesian'))
}

for path in PATHS.values():
    os.makedirs(path, exist_ok=True)
```


## Source cell 5 (original zero-based index 4)


Reference only; not executed.

```python
def get_dataset_paths():
    base_train = os.path.join(PATHS['matlab_workspaces'], 'PDMS2', 'Dynamic', 'Spiral', 'Set3')
    base_test = os.path.join(PATHS['matlab_workspaces'], 'PDMS2', 'Dynamic', 'Tests', 'Set3')
    return {
        'train': [
            os.path.join(base_train, f'Dynamic_PDMS2_Spiral_Set3_{s}.mat')
            for s in ['0.30_0.10', '0.20_0.20', '0.22_0.18','0.15_0.16', '0.18_0.14', '0.25_0.14', '0.28_0.12', '0.12_0.11']
        ],
        'test': [
            os.path.join(base_test, f'Dynamic_PDMS2_{t}_Set3_0.20_0.10.mat')
            for t in ['Spiral', 'Inf', 'Bend', 'Sine', 'Line', 'Squarewave', 'Step']
        ]
    }
DATASETS = get_dataset_paths()
```


## Source cell 8 (original zero-based index 7)


In [ ]:
class Actor(nn.Module):
    def __init__(self, actor_input_dim, action_dim, action_max, action_min, actor_hidden, input_min, input_max):
        super().__init__()
        self.register_buffer('action_max', torch.tensor(action_max, dtype=torch.float32))
        self.register_buffer('action_min', torch.tensor(action_min, dtype=torch.float32))
        self.register_buffer('input_min', torch.tensor(input_min, dtype=torch.float32))
        self.register_buffer('input_max', torch.tensor(input_max, dtype=torch.float32))
        layers = []
        for size in actor_hidden:
            layers.extend([nn.Linear(actor_input_dim, size), nn.LayerNorm(size), nn.ReLU()])
            actor_input_dim = size
        layers.append(nn.Linear(actor_input_dim, action_dim))
        self.net = nn.Sequential(*layers).to(device)
        self.apply(self._init_weights)
   
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(m.bias)
   
    def forward(self, x):
        x_norm = 2 * (x - self.input_min) / (self.input_max - self.input_min) - 1
        act_norm = torch.tanh(self.net(x_norm))
        act = 0.5 * (act_norm + 1) * (self.action_max - self.action_min) + self.action_min
        return act


## Source cell 10 (original zero-based index 9)


In [ ]:
class Critic(nn.Module):
    def __init__(self, critic_input_dim, hidden_sizes, input_min, input_max):
        super().__init__()
        self.register_buffer('input_min', torch.tensor(input_min, dtype=torch.float32))
        self.register_buffer('input_max', torch.tensor(input_max, dtype=torch.float32))
        layers = []
        for size in hidden_sizes:
            layers.extend([nn.Linear(critic_input_dim, size), nn.LayerNorm(size), nn.ReLU()])
            critic_input_dim = size
        layers.append(nn.Linear(critic_input_dim, 1))
        self.q_net = nn.Sequential(*layers).to(device)
        self.apply(self._init_weights)
   
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(m.bias)
   
    def forward(self, x):
        x_norm = 2 * (x - self.input_min) / (self.input_max - self.input_min) - 1
        return self.q_net(x_norm)


## Source cell 12 (original zero-based index 11)


In [ ]:
class ReplayBuffer:
    def __init__(self, state_dim, action_dim, capacity=int(1e6)):
        self.capacity = capacity
        self.idx = 0
        self.full = False
        
        self.states = torch.zeros((capacity, state_dim), device=device, dtype=torch.float32)
        self.actions = torch.zeros((capacity, action_dim), device=device, dtype=torch.float32)
        self.rewards = torch.zeros(capacity, device=device, dtype=torch.float32)

    def add(self, state, action, reward):
        idx = self.idx
        
        self.states[idx] = torch.as_tensor(state, device=device, dtype=torch.float32)
        self.actions[idx] = torch.as_tensor(action, device=device, dtype=torch.float32)
        self.rewards[idx] = torch.as_tensor(reward, device=device, dtype=torch.float32)
        
        self.idx = (self.idx + 1) % self.capacity
        self.full = self.full or (self.idx == 0)

    def sample(self, batch_size):
        max_idx = self.capacity if self.full else self.idx
        indices = torch.randint(0, max_idx, (batch_size,), device=device)
        
        return (
            self.states[indices],
            self.actions[indices],
            self.rewards[indices],
        )


## Source cell 19 (original zero-based index 18)


In [ ]:
class DDPG:
    def __init__(self, env, actor_hidden=(400, 300), critic_hidden=(400, 300),
                 lr_actor=1e-4, lr_critic=1e-3, gamma=0.99, tau=0.005, batch_size=128,
                 noise_init=0.1, noise_decay=0.995, noise_min=0.01):

        self.env, self.gamma, self.tau, self.batch_size = env, gamma, tau, batch_size
        self.ou_increase = 0.001
        self.video_dir = PATHS['renders_train']
        self.plots_points_dir = PATHS['plots_generated_points']
        
        self.generated_points, self.all_ep_tips = [], []
        self.noise_scale, self.noise_decay, self.noise_min = noise_init, noise_decay, noise_min

        self.state_dim, self.action_dim = env.observation_space.shape[0], env.action_space.shape[0]           
        
        act_low, act_high = env.action_space.low, env.action_space.high
        state_low, state_high = env.observation_space.low, env.observation_space.high
        
        self.action_low_t = torch.tensor(act_low, dtype=torch.float32, device=device)
        self.action_high_t = torch.tensor(act_high, dtype=torch.float32, device=device)
        self.action_range = torch.tensor((act_high - act_low) / 6.0, device=device) 

        # --- Networks ---
        self.actor = Actor(self.state_dim, self.action_dim, act_high, act_low, actor_hidden, state_low, state_high).to(device)
        self.critic = Critic(self.state_dim + self.action_dim, critic_hidden, 
                             np.concatenate([state_low, act_low]), np.concatenate([state_high, act_high])).to(device)

        self.actor_optim = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_optim = optim.Adam(self.critic.parameters(), lr=lr_critic, weight_decay=1e-2)
        
        # Ensure your ReplayBuffer returns: states, actions, rewards, next_states, dones
        self.buffer = ReplayBuffer(self.state_dim, self.action_dim)
        
        self.history = {k: [] for k in ['returns', 'success_rates', 'step_term', 
                                        'critic_losses', 'actor_losses', 'pos_norm', 'dir_norm', 'v_norm', 'w_norm',
                                        'r_pos', 'r_dir', 'r_v', 'r_w']}

    def store_history(self, ep_reward, ep_success, s_term, cl_loss, al_loss, metrics):
        h = self.history
        h['returns'].append(ep_reward)
        h['success_rates'].append(ep_success)
        h['step_term'].append(s_term)
        h['critic_losses'].append(np.mean(cl_loss) if cl_loss else 0.0)
        h['actor_losses'].append(np.mean(al_loss) if al_loss else 0.0)
        for k in ['pos_norm', 'dir_norm', 'v_norm', 'w_norm', 'r_pos', 'r_dir', 'r_v', 'r_w']:
            h[k].append(np.mean(metrics[k]) if metrics[k] else 0.0)

    def act(self, state, training=True):
        with torch.no_grad():
            state_t = torch.as_tensor(state, dtype=torch.float32, device=device)
            action = self.actor(state_t)
            if training:
                action += torch.normal(0.0, self.action_range * self.noise_scale)
            return torch.clamp(action, self.action_low_t, self.action_high_t).cpu().numpy()

    def update(self):
        states, actions, rewards = self.buffer.sample(self.batch_size)
        rewards = rewards.unsqueeze(1)

        # --- Critic Update ---
        with torch.no_grad():
            target_q = rewards

        current_q = self.critic(torch.cat([states, actions], dim=1))
        critic_loss = nn.MSELoss()(current_q, target_q)
        
        self.critic_optim.zero_grad()
        critic_loss.backward()
        self.critic_optim.step()
        
        # --- Actor Update ---
        for p in self.critic.parameters(): p.requires_grad = False
        actor_loss = -self.critic(torch.cat([states, self.actor(states)], dim=1)).mean()
        self.actor_optim.zero_grad()
        actor_loss.backward()
        self.actor_optim.step()
        for p in self.critic.parameters(): p.requires_grad = True
            
        return critic_loss.item(), actor_loss.item()
    
    def train(self, episodes=200, steps=500):
        best_reward = float('-inf')

        for ep in range(1, episodes + 1):

            # # --- Curriculum Learning scaling ---
            # curriculum_length = 100.0
            # target_sigma = self.env.action_space.high / 3.0
            # self.env.ou_noise.sigma = np.minimum((ep / curriculum_length) * target_sigma, target_sigma)
            # # -------------------------------------------

            # action0 = np.array([0.0, 0.0, 0.30, 1.0, 0.0, 0.0, 0.0])        # 0 deg Pull
            action0 = None
            state = self.env.reset(action0)

            ep_reward, ep_success, step_term = 0, 0, 0
            cl_loss, al_loss, ep_p_tips = [], [], [] 
            
            # Match dictionary keys exactly to the environment's returned info keys
            metrics_keys = ['pos_norm', 'dir_norm', 'v_norm', 'w_norm', 'r_pos', 'r_dir', 'r_v', 'r_w']
            metrics = {k: [] for k in metrics_keys}
            run = {k: 0.0 for k in metrics_keys}
            run.update({'cl': 0.0, 'al': 0.0})
            upd_cnt = 0
                
            with tqdm(total=steps, desc=f'Ep {ep}/{episodes}', unit='step') as pbar:
                for step in range(steps):
                    action = self.act(state, training=True)
                    next_state, reward, info = self.env.step(action)
                    
                    # Assume info dict keys match what MatlabEnv step() returns
                    if info.get('step_term', False):
                        state = self.env.reset(action0)
                        step_term += 1
                    else:
                        self.buffer.add(state, action, reward)
                        ep_p_tips.append(self.env.p_target) 
                        ep_success += int(info.get('is_success', False))
                            
                        for k in metrics_keys: 
                            val = info.get(k, 0.0)
                            metrics[k].append(val)
                            run[k] += val
                        
                        state = next_state
                        ep_reward += reward

                        if self.buffer.idx > self.batch_size:
                            cl, al = self.update()
                            cl_loss.append(cl); al_loss.append(al)
                            run['cl'] += cl; run['al'] += al
                            upd_cnt += 1

                            s_div = step + 1
                            pbar.set_postfix_str(
                                f"CL: {run['cl']/upd_cnt:.3f} | AL: {run['al']/upd_cnt:.3f} | Rew: {ep_reward:.1f} | Succ: {(ep_success/steps)*100:.1f} | "
                                f"PE: {(run['pos_norm']/s_div)*1000:.1f} | RP: {run['r_pos']/s_div:.2f} | "
                                f"DE: {run['dir_norm']/s_div:.2f} | RD: {run['r_dir']/s_div:.2f} | "
                                f"VE: {(run['v_norm']/s_div)*100:.1f} | RV: {run['r_v']/s_div:.2f} | "
                                f"WE: {(run['w_norm']/s_div)*100:.1f} | RW: {run['r_w']/s_div:.2f} | "
                                f"S_T: {step_term}"
                            )

                    pbar.update(1)

            self.all_ep_tips.extend(ep_p_tips)
            self.noise_scale = max(self.noise_scale * self.noise_decay, self.noise_min)
            self.store_history(ep_reward, ep_success, step_term, cl_loss, al_loss, metrics)

            torch.save(self.actor.state_dict(), os.path.join(PATHS['last_weights'], 'actor_dict.pth'))
            torch.save(self.critic.state_dict(), os.path.join(PATHS['last_weights'], 'critic_dict.pth'))
                            
            if ep > episodes - 50:
                torch.save(self.actor.state_dict(), os.path.join(PATHS['latest_models'], f'actor_ep_{ep}.pth'))
                
            # Save visuals
            if ep % 250 == 0 or ep in [10, 50, 100]:
                save_path = os.path.join(PATHS['plots_generated_points'], f'random_points_up_to_ep_{ep}.png')
                plot_3d_random_points(self.all_ep_tips, save_path, f'Cumulative Random Generated Points up to Episode {ep}')
            
            if ep % 250 == 0:
                print(f"Rendering video for episode {ep}...")
                render_file = DATASETS['train'][0]
                video_path_main = os.path.join(self.video_dir, f'episode_{ep}_render_main.mp4')
                video_path_setup = os.path.join(self.video_dir, f'episode_{ep}_render_setup.mp4')
                render_movement(self, self.env, render_file, video_path_main, video_path_setup, show=False)

        return self.history, self.actor, self.critic


## Source cell 21 (original zero-based index 20)


Reference only; not executed.

```python
matter = 'PDMS2'
mag_set = 'Set3'
dataset_path = os.path.join(current_dir, 'Catheter_DL_Dataset.mat')
env = MatlabEnv_dataset(matter=matter, mag_set=mag_set, dataset_path= dataset_path)
```


## Source cell 23 (original zero-based index 22)


Reference only; not executed.

```python
%%time

agent = DDPG(
    env,
    actor_hidden=(400, 300),
    critic_hidden=(400, 300),
    lr_actor=1e-4,
    lr_critic=5e-4,
    gamma=0.99, 
    tau=0.005,
    batch_size=256,
    noise_init=0.5,
    noise_decay=0.9,
    noise_min=0.01
)

# The train function now returns more metrics
(history, actor_model, critic_model) = agent.train(episodes=2000, steps=500)

env.close()
```
